# Iceberg v3: deletion vectors and row lineage

Two v3 features that change how row-level changes work, both usable in this lab (Spark 4.1.3 + Iceberg 1.12.0 + Trino 483 + the HMS Iceberg REST catalog):

* **Deletion vectors (DVs).** In a v2 table a merge-on-read `DELETE` writes a *position-delete Parquet file*. In v3 it writes a compact **bitmap in a Puffin file** attached to the data file.
* **Row lineage.** Every row has a stable `_row_id` and a `_last_updated_sequence_number` (metadata columns). That gives you cheap *"what changed since I last looked?"* queries without a CDC pipeline.

Iceberg 1.12.0 hardens both (release notes: fixes for row-lineage sequence-number inheritance, DVs in the same Puffin file, commit validation with several DVs, plus a scan-based action and the `remove-dangling-deletes` rewrite option to clean up delete files).

You will compare v2 and v3 deletes, read lineage columns, build an incremental "changes since" query (and find out where it cannot see deletes), and compact the table.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))
from lab_utils import get_spark, trino, try_sql, CATALOG

spark = get_spark("iceberg-dv-lineage")
print("Spark", spark.version)
NS = f"{CATALOG}.iceberg12"
_ = spark.sql(f"CREATE NAMESPACE IF NOT EXISTS {NS}")
MOR = "'write.delete.mode'='merge-on-read','write.update.mode'='merge-on-read','write.merge.mode'='merge-on-read'"

## 1. The same deletes on a v2 and a v3 table

In [ ]:
for name, fv in (("orders_v2", 2), ("orders_v3", 3)):
    spark.sql(f"DROP TABLE IF EXISTS {NS}.{name}")
    spark.sql(f"CREATE TABLE {NS}.{name} (order_id BIGINT, status STRING, amount DOUBLE) USING iceberg TBLPROPERTIES ('format-version'='{fv}', {MOR})")
    spark.sql(f"INSERT INTO {NS}.{name} SELECT id, 'new', id * 1.5 FROM range(1, 101)")
    spark.sql(f"DELETE FROM {NS}.{name} WHERE order_id IN (5, 6, 7)")
    print(f"== {name} (format v{fv}): {spark.sql(f'SELECT count(*) FROM {NS}.{name}').collect()[0][0]} rows visible")
    spark.sql(f"SELECT content, file_format, record_count, referenced_data_file IS NOT NULL AS bound_to_one_data_file FROM {NS}.{name}.delete_files").show()

* v2: `file_format = PARQUET`, a position-delete file. v3: `file_format = PUFFIN`, a **deletion vector** bound to exactly one data file.
* Readers (Spark and Trino here) apply either transparently. Trino 483 sees the v3 table and its DV:

In [ ]:
print(trino("SELECT count(*) AS rows_visible_in_trino FROM iceberg.iceberg12.orders_v3").to_string(index=False))
spark.sql(f"SELECT operation, summary['added-dvs'] AS added_dvs, summary['added-position-delete-files'] AS added_pos_delete_files FROM {NS}.orders_v3.snapshots ORDER BY committed_at").show()

## 2. Row lineage: `_row_id` and `_last_updated_sequence_number`
They are *metadata columns*: select them explicitly. `_row_id` identifies the row for its whole life; `_last_updated_sequence_number` is the commit sequence number that last wrote it.

In [ ]:
T = f"{NS}.orders_v3"
spark.sql(f"SELECT order_id, status, _row_id, _last_updated_sequence_number FROM {T} WHERE order_id IN (4, 8) ORDER BY order_id").show()
spark.sql(f"UPDATE {T} SET status = 'shipped' WHERE order_id = 8")
spark.sql(f"SELECT order_id, status, _row_id, _last_updated_sequence_number FROM {T} WHERE order_id IN (4, 8) ORDER BY order_id").show()

Order 8 keeps its `_row_id` but its `_last_updated_sequence_number` moved forward; order 4 is untouched.

## 3. Incremental processing with a high-water mark
Pattern: remember the highest sequence number you processed; next time read only rows with a **larger** one. New and updated rows qualify; no CDC tool needed.

Gotcha in this stack: `max()` over a lineage column fails in Iceberg 1.12.0 with Spark's aggregate push-down (`Cannot find field '_last_updated_sequence_number'`). The cell shows it, then uses the workaround (switch the push-down off).

In [ ]:
q_max = f"SELECT max(_last_updated_sequence_number) FROM {T}"
try_sql(spark, q_max, "max() over a lineage column (aggregate push-down on)")
spark.conf.set("spark.sql.iceberg.aggregate-push-down.enabled", "false")
hwm = spark.sql(q_max).collect()[0][0]
print("high-water mark:", hwm)

In [ ]:
# --- later: more writes happen ---------------------------------------------------------
spark.sql(f"UPDATE {T} SET status = 'shipped' WHERE order_id IN (20, 21)")
spark.sql(f"INSERT INTO {T} VALUES (101, 'new', 10.0), (102, 'new', 20.0)")
spark.sql(f"DELETE FROM {T} WHERE order_id = 50")

# --- the incremental read ---------------------------------------------------------------
spark.sql(f"""SELECT order_id, status, _row_id, _last_updated_sequence_number
              FROM {T} WHERE _last_updated_sequence_number > {hwm} ORDER BY order_id""").show()

The two updates and the two inserts show up. The **delete of order 50 does not**: a deleted row no longer exists, so it has no lineage to read.
Two ways to catch deletes:

* **Snapshot diff** with time travel: compare keys now against keys at the previous snapshot (below).
* Iceberg's `create_changelog_view` is the usual tool, but changelog scans do not support delete files in 1.12.0, so it cannot be used on tables with merge-on-read deletes or DVs (shown below).

In [ ]:
prev = spark.sql(f"SELECT snapshot_id FROM {T}.snapshots ORDER BY committed_at DESC LIMIT 1 OFFSET 3").collect()[0][0]
print("comparing against snapshot", prev)
spark.sql(f"""SELECT order_id AS deleted_order_id FROM {T} VERSION AS OF {prev}
              EXCEPT SELECT order_id FROM {T}""").show()

In [ ]:
first = spark.sql(f"SELECT snapshot_id FROM {T}.snapshots ORDER BY committed_at LIMIT 1").collect()[0][0]
try_sql(spark, f"CALL {CATALOG}.system.create_changelog_view(table => 'iceberg12.orders_v3', options => map('start-snapshot-id','{first}'), changelog_view => 'orders_changes')",
        "create_changelog_view (creates the view)")
try_sql(spark, "SELECT count(*) FROM orders_changes", "read the changelog view on a table with DVs")

## 4. Compaction removes the DVs, and row ids survive
`rewrite_data_files` merges the data files with their deletes. The `remove-dangling-deletes` option (documented for the procedure) also drops delete files that no longer apply.
Lineage must survive: Iceberg 1.12 fixed several lineage bugs around rewrites, so check it on your own tables.

In [ ]:
ids = "order_id IN (8, 101)"
before = spark.sql(f"SELECT order_id, _row_id FROM {T} WHERE {ids} ORDER BY order_id").collect()
spark.sql(f"SELECT count(*) AS delete_files FROM {T}.delete_files").show()
r = spark.sql(f"""CALL {CATALOG}.system.rewrite_data_files(table => 'iceberg12.orders_v3',
        options => map('rewrite-all','true','remove-dangling-deletes','true'))""").collect()[0]
print(f"rewrote {r.rewritten_data_files_count} data files into {r.added_data_files_count}, removed {r.removed_delete_files_count} delete files")
spark.sql(f"SELECT count(*) AS delete_files_after FROM {T}.delete_files").show()
after = spark.sql(f"SELECT order_id, _row_id FROM {T} WHERE {ids} ORDER BY order_id").collect()
print("row ids unchanged by compaction:", before == after)
print("rows:", spark.sql(f"SELECT count(*) FROM {T}").collect()[0][0])

## Notes and limits
* v3 tables need readers that understand v3. Spark 4.x and Trino 483 do here; check other engines before upgrading production tables from v2.
* `_last_updated_sequence_number` tells you *when* a row last changed in commit order, not *what* changed in it.
* In this lab the Iceberg REST catalog is embedded in Hive Metastore 4.2.1 (it embeds Iceberg 1.9.1); it accepted v3 tables, variants and DVs in the tests behind these notebooks.

In [ ]:
CLEANUP = False   # set True to drop the tables created here
if CLEANUP:
    for t in ["orders_v2", "orders_v3"]:
        spark.sql(f"DROP TABLE IF EXISTS {NS}.{t}")